# EDA silver layer

In [0]:

df = spark.sql("SELECT * FROM marathos.bronze.raw_marathos")

display(df)

In [0]:
df_country_code = spark.sql("FROM marathos.bronze.raw_country_code")
df_country_code.display()

In [0]:
from Marathos.utils.utils import rename_columns_to_snake_case
df = rename_columns_to_snake_case(df)
df.display()

In [0]:
from Marathos.utils.utils import rename_columns_to_snake_case
df = rename_columns_to_snake_case(df)
df.display()

In [0]:
from pyspark.sql.functions import regexp_extract, split, when, col, lit, get, size

df_valid = (
    df
    .withColumn(
        "event_unit",
        regexp_extract(col("event_distance/length"), r"(km|mi|h)", 1)
    )
    .withColumn(
        "performance_clean",
        regexp_extract(col("athlete_performance"), r"^([\d:\.]+)", 1)
    )
    .withColumn(
        "performance_hours",
        when(
            col("event_unit").isin("km", "mi"),
            when(
                size(split(col("performance_clean"), ":")) == 3,  # H:mm:ss
                get(split(col("performance_clean"), ":"), 0).cast("float") +
                get(split(col("performance_clean"), ":"), 1).cast("float") / 60 +
                get(split(col("performance_clean"), ":"), 2).cast("float") / 3600
            ).when(
                size(split(col("performance_clean"), ":")) == 2,  # H:mm
                get(split(col("performance_clean"), ":"), 0).cast("float") +
                get(split(col("performance_clean"), ":"), 1).cast("float") / 60
            ).otherwise(lit(None))
        ).otherwise(lit(None))
    )
    .withColumn(
        "performance_km",
        when(
            col("event_unit") == "h",
            col("performance_clean").cast("float")
        ).otherwise(lit(None))
    )
    .drop("athlete_performance")
)

df_valid.select(
    "event_distance/length", "event_unit", "performance_hours", "performance_km"
).display()

In [0]:
df_valid.select("athlete_country").display()

In [0]:
df_with_with_country = (
    df_valid
    .join(
        df_country_code.select("iso3", "country_common"),
        df_valid["athlete_country"] == df_country_code["iso3"],
        how="left" 
    )
    .drop("iso3") 
    .withColumnRenamed("country_common", "athlete_country_name")
)


split race name from country code

In [0]:
from pyspark.sql.functions import regexp_extract, regexp_replace, col

df_country_eve = (
    df_with_with_country

    .withColumn("event_country", regexp_extract(col("event_name"), r"\(([^)]+)\)$", 1))

    # Ta bort landet + parentes från event-namnet
    .withColumn("event_name_clean", regexp_replace(col("event_name"), r"\s*\([^)]+\)$", ""))
)

df_country_eve.display()
